# 05. Final Load Prep — Master Export (Tableau Ready)
**Project:** Logistics & Delivery Delays Analysis (Olist)
**Approach:** Single source of truth with 2 CSV files for all 4 dashboards


In [16]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

PATH_CLEANED = "../data/processed/olist_cleaned_data.csv"
PATH_TABLEAU = "../data/processed/tableau_ready/"
os.makedirs(PATH_TABLEAU, exist_ok=True)

df = pd.read_csv(PATH_CLEANED)
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_year'] = df['order_purchase_timestamp'].dt.year
df['order_quarter'] = df['order_purchase_timestamp'].dt.to_period('Q').dt.strftime('Q%q-%Y')
df['is_bad_review'] = df['review_score'] == 1
print(f"Base dataset loaded. Shape: {df.shape}")

def clean_colnames(dataframe):
    df_out = dataframe.copy()
    df_out.columns = [col.replace('_', ' ').title() for col in df_out.columns]
    return df_out


Base dataset loaded. Shape: (109657, 46)


## Data Strategy: 2 Master CSVs

Instead of 6 separate CSVs, we export **2 complementary master datasets** that contain ALL data needed for all 4 dashboards:

| Dashboard | Required Data | Source |
|-----------|---------------|--------|
| D1 - Executive Pulse | Time-series KPIs | agg_summary (groupby year + quarter) |
| D2 - Geographic Risk | State metrics + coordinates | agg_summary (groupby seller_state) + orders_master (lat/lng) |
| D3 - Seller Performance | Seller health scores | agg_summary (groupby seller_id) |
| D4 - Customer Experience | Order scatter plots | orders_master (30K sample) |

**Advantages:**
- Single source of truth
- Flexible aggregation in Tableau
- Cleaner data pipeline
- No redundant exports


## 5. Master Export Layer — Enhanced for Geographic Analysis

### Overview
Two complementary datasets optimized for all 4 dashboards:
- **File 1 (orders_master):** Row-level with seller/customer coordinates
- **File 2 (agg_summary):** Aggregated metrics with avg lat/lng for geographic mapping

**Key Addition:** Latitude & Longitude included for D2 Geographic Risk Dashboard
**Join Keys:** Seller State + Order Year


### File 1: Orders Master (30K Row-level with Coordinates)
**Purpose:** Scatter plots, trend filters, order drill-down, geographic mapping  
**Key Columns:** lat/lng from seller & customer for D2 mapping  
**Output:** `olist_orders_master.csv`


In [17]:
# FILE 1: Orders Master with geographic coordinates for D2 dashboard
orders_master = df[[
    'order_id', 'order_year', 'order_quarter',
    'order_purchase_timestamp', 'delivery_delay',
    'actual_delivery_days', 'estimated_delivery_days',
    'review_score', 'is_late', 'is_bad_review',
    'seller_state', 'seller_city', 'seller_id',
    'seller_lat', 'seller_lng',
    'customer_state', 'customer_city',
    'customer_lat', 'customer_lng',
    'shipping_route', 'haversine_distance_km'
]].copy()

orders_master['delivery_status'] = orders_master['is_late'].map(
    {True: 'Late', False: 'On Time'})
orders_master['review_tier'] = orders_master['is_bad_review'].map(
    {True: '1-2 Star (Bad)', False: '3-5 Star (Good)'})

orders_master = orders_master.drop(columns=['is_late', 'is_bad_review'])
orders_master = orders_master.sample(n=min(30000, len(orders_master)), random_state=42)

output_file_1 = os.path.join(PATH_TABLEAU, 'olist_orders_master.csv')
clean_colnames(orders_master).to_csv(output_file_1, index=False)

print(f'✓ File 1 exported: olist_orders_master.csv | {orders_master.shape}')

✓ File 1 exported: olist_orders_master.csv | (30000, 21)


### File 2: Aggregated Summary (Performance Metrics)
**Purpose:** Risk metrics, route performance, seller health scores  
**Note:** Geographic data for D2 comes from `orders_master` (row-level coordinates)  
**Output:** `olist_aggregated_summary.csv`  

**Why No Geographic Coordinates Here?**
- Averaging customer coordinates across multiple orders creates meaningless "phantom" points
- D2 dashboard uses `orders_master` directly for accurate geographic mapping
- agg_summary focuses on business metrics (delays, reviews, risk) that benefit from aggregation

In [18]:
# FILE 2: Aggregated summary with performance metrics (no geographic averaging)
agg_summary = df.groupby(
    ['seller_state', 'customer_state', 'order_year', 'order_quarter', 'seller_id']
).agg(
    total_orders       = ('order_id',                'count'),
    late_orders        = ('is_late',                  'sum'),
    avg_delay_days     = ('delivery_delay',           'mean'),
    avg_review_score   = ('review_score',             'mean'),
    bad_reviews        = ('is_bad_review',            'sum'),
    avg_distance_km    = ('haversine_distance_km',    'mean'),
).reset_index()

# Calculate derived metrics
agg_summary['late_rate_pct']    = (agg_summary['late_orders']  / agg_summary['total_orders'] * 100).round(2)
agg_summary['bad_review_rate']  = (agg_summary['bad_reviews']  / agg_summary['total_orders'] * 100).round(2)
agg_summary['avg_delay_days']   = agg_summary['avg_delay_days'].round(2)
agg_summary['avg_review_score'] = agg_summary['avg_review_score'].round(3)

# Risk classification
agg_summary['high_risk_seller'] = (agg_summary['late_rate_pct'] > 13).map(
    {True: 'High Risk', False: 'Normal'})

# Shipping route
agg_summary['shipping_route']   = agg_summary['seller_state'] + ' -> ' + agg_summary['customer_state']

output_file_2 = os.path.join(PATH_TABLEAU, 'olist_aggregated_summary.csv')
clean_colnames(agg_summary).to_csv(output_file_2, index=False)

print(f'✓ File 2 exported: olist_aggregated_summary.csv | {agg_summary.shape}')
print('\n✅ Export complete!')
print('\nTableau Setup Guide:')
print('  D1 (Executive Pulse):        Use agg_summary → Group by Year + Quarter')
print('  D2 (Geographic Risk):        Use orders_master → Map seller/customer coordinates')
print('  D3 (Seller Performance):     Use agg_summary → Group by Seller ID')
print('  D4 (Customer Experience):    Use orders_master → Scatter plot all orders')

✓ File 2 exported: olist_aggregated_summary.csv | (32860, 15)

✅ Export complete!

Tableau Setup Guide:
  D1 (Executive Pulse):        Use agg_summary → Group by Year + Quarter
  D2 (Geographic Risk):        Use orders_master → Map seller/customer coordinates
  D3 (Seller Performance):     Use agg_summary → Group by Seller ID
  D4 (Customer Experience):    Use orders_master → Scatter plot all orders
